# Klasifikasi Sampah - Fine-tuning Vision Transformer
Notebook mandiri (semua dalam satu file). Alurnya:
1. Konfigurasi dan import
2. Split data (per domain website, supaya validasi jujur)
3. Model transformer pretrained (timm) + dataloader
4. Training
5. Evaluasi (macro-F1, confusion matrix)
6. (Nanti) Ekspor ke format `submission/` untuk `evaluate.py`

Letakkan notebook ini sefolder dengan `evaluate.py`, folder `train/`, dan `train_sources.csv`.

In [ ]:
# !pip install -q timm   # buka komentar ini kalau di Colab

import csv
import math
import random
import shutil
import time
from collections import Counter, defaultdict
from pathlib import Path
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import timm
import torch
import torch.nn as nn
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from evaluate import CLASSES, metrics  # fungsi metrik yang sama dengan penilai resmi

# ===== Konfigurasi (ubah di sini) =====
NAME = "vit_small_patch16_224.augreg_in21k_ft_in1k"   # alternatif: swin_tiny_patch4_window7_224, vit_base_patch16_224.augreg_in21k_ft_in1k
IMG = 224
DATA = "train"                 # folder data asli: train/<kelas>/*.jpg
SOURCES = "train_sources.csv"  # info URL sumber gambar (untuk split per domain)
TRAIN_DIR, VAL_DIR = "gtrain", "gval"
VAL_FRAC = 0.15

EPOCHS = 15
BATCH = 32
LR = 5e-5
WD = 0.05
USE_MIXUP = True               # Mixup/CutMix, matikan kalau ingin lebih sederhana
SEED = 42
BEST_PATH = "best_vit.pth"

EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
amp = device == "cuda"
print("device:", device)

## 1. Split data
Gambar dari website yang sama sering mirip. Kalau dibagi acak per gambar, gambar mirip bisa masuk train **dan** validasi sehingga skor jadi terlalu bagus. Di sini gambar dikelompokkan per domain, lalu satu domain utuh masuk ke train atau validasi saja.
Split hanya dibuat sekali. Hapus `gtrain/` dan `gval/` kalau mau split ulang.

In [ ]:
def domain(url):
    host = urlparse(url).netloc.lower()
    return host[4:] if host.startswith("www.") else host


def make_split():
    if Path(VAL_DIR).exists():
        print(f"{VAL_DIR}/ sudah ada, dipakai ulang")
        return

    groups = defaultdict(list)
    if Path(SOURCES).exists():
        with open(SOURCES, encoding="utf-8") as f:
            for r in csv.DictReader(f):
                if (Path(DATA) / r["file"]).exists():
                    key = domain(r["page_url"]) or domain(r["image_url"]) or r["file"]
                    groups[key].append(r["file"])
    else:
        print(f"{SOURCES} tidak ditemukan -> split acak per gambar")
        for p in Path(DATA).glob("*/*"):
            if p.suffix.lower() in EXTS:
                groups[str(p)].append(f"{p.parent.name}/{p.name}")

    total = sum(len(v) for v in groups.values())
    target = int(total * VAL_FRAC)
    keys = sorted(groups)
    random.Random(SEED).shuffle(keys)

    val, train = [], []
    for k in keys:
        if len(val) + len(groups[k]) <= target:
            val += groups[k]
        else:
            train += groups[k]

    for files, dst in ((train, Path(TRAIN_DIR)), (val, Path(VAL_DIR))):
        for rel in files:
            out = dst / rel
            out.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(Path(DATA) / rel, out)

    print("grup:", len(groups), "| train:", len(train), "| val:", len(val))
    print("val per kelas  :", dict(Counter(p.split("/")[0] for p in val)))
    print("train per kelas:", dict(Counter(p.split("/")[0] for p in train)))


make_split()

## 2. Model dan dataloader
`timm.create_model(..., pretrained=True, num_classes=5)` memuat bobot pretrained dan memasang head baru 5 kelas. Mean/std normalisasi diambil dari konfigurasi model tersebut.

In [ ]:
model = timm.create_model(NAME, pretrained=True, num_classes=len(CLASSES))
cfg = timm.data.resolve_data_config({}, model=model)
MEAN, STD = cfg["mean"], cfg["std"]
print("mean/std:", MEAN, STD)
print("parameter:", round(sum(p.numel() for p in model.parameters()) / 1e6, 1), "juta")

norm = transforms.Normalize(MEAN, STD)
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    norm,
    transforms.RandomErasing(p=0.25),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    norm,
])

tr_ds = datasets.ImageFolder(TRAIN_DIR, train_tf)
va_ds = datasets.ImageFolder(VAL_DIR, val_tf)
assert tr_ds.classes == CLASSES, f"urutan kelas beda: {tr_ds.classes}"
print(tr_ds.classes, len(tr_ds), len(va_ds))

tr_dl = DataLoader(tr_ds, BATCH, shuffle=True, drop_last=True, num_workers=2, pin_memory=amp)
va_dl = DataLoader(va_ds, BATCH, num_workers=2, pin_memory=amp)

## 3. Training

In [ ]:
model.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
steps = EPOCHS * len(tr_dl)
warm = len(tr_dl)  # warmup 1 epoch, lalu cosine decay
sched = torch.optim.lr_scheduler.LambdaLR(
    opt,
    lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / (steps - warm))),
)
scaler = torch.amp.GradScaler("cuda", enabled=amp)

if USE_MIXUP:
    mix = Mixup(mixup_alpha=0.2, cutmix_alpha=1.0, prob=0.5, switch_prob=0.5,
                label_smoothing=0.1, num_classes=len(CLASSES))
    loss_fn = SoftTargetCrossEntropy()
else:
    mix = None
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)


@torch.no_grad()
def predict(net, loader):
    net.eval()
    y_true, y_pred = [], []
    for x, y in loader:
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=amp):
            out = net(x.to(device))
        y_pred += out.argmax(1).cpu().tolist()
        y_true += y.tolist()
    return y_true, y_pred


best = 0.0
hist = {"loss": [], "f1": [], "acc": []}

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    total = 0.0
    for x, y in tr_dl:
        x, y = x.to(device), y.to(device)
        if mix:
            x, y = mix(x, y)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=amp):
            loss = loss_fn(model(x), y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        total += loss.item()

    m = metrics(*predict(model, va_dl))
    hist["loss"].append(total / len(tr_dl))
    hist["f1"].append(m["macro_f1"])
    hist["acc"].append(m["accuracy"])
    print(f"epoch {ep:02d} loss={hist['loss'][-1]:.4f} macro_f1={m['macro_f1']:.4f} "
          f"acc={m['accuracy']:.4f} ({time.time() - t0:.0f}s)")

    if m["macro_f1"] > best:
        best = m["macro_f1"]
        torch.save(model.state_dict(), BEST_PATH)

print("best macro_f1:", round(best, 4))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(hist["loss"], marker="o"); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(hist["f1"], marker="o", label="macro-F1")
ax[1].plot(hist["acc"], marker="o", label="akurasi")
ax[1].set_title("validasi"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout()
plt.show()

## 4. Evaluasi model terbaik
Muat bobot epoch terbaik, lalu lihat F1 per kelas dan confusion matrix.

In [ ]:
model.load_state_dict(torch.load(BEST_PATH, map_location=device, weights_only=True))
m = metrics(*predict(model, va_dl))

print(f"Macro-F1: {m['macro_f1']:.4f} | Akurasi: {m['accuracy']:.4f}")
for c, f in m["per_class_f1"].items():
    print(f"  F1 {c:<15} {f:.4f}")

cm = m["confusion_matrix"]
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
ax.set_xlabel("prediksi"); ax.set_ylabel("label asli")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i][j], ha="center", va="center",
                color="white" if cm[i][j] > max(map(max, cm)) / 2 else "black")
plt.tight_layout()
plt.show()

## 5. (Nanti) Ekspor ke format submission
Sel ini menulis `submission/model.py` dan menyalin bobot terbaik ke `submission/model.pth`, lalu menjalankan penilai resmi. Jalankan setelah model sudah memuaskan.

In [ ]:
sub = Path("submission")
sub.mkdir(exist_ok=True)

model_py = f"""import timm
from torchvision import transforms

NAME = "{NAME}"
IMG = {IMG}


def get_model():
    return timm.create_model(NAME, pretrained=False, num_classes=5)


def get_transform():
    return transforms.Compose([
        transforms.Resize((IMG, IMG)),
        transforms.ToTensor(),
        transforms.Normalize({tuple(MEAN)}, {tuple(STD)}),
    ])
"""
(sub / "model.py").write_text(model_py)
shutil.copy2(BEST_PATH, sub / "model.pth")
print((sub / "model.py").read_text())

In [ ]:
!python evaluate.py --submission ./submission --data ./gval